

# Model Validation & Risk Assessment

## Objective
Validate a credit risk model by checking:
- Performance consistency
- Overfitting
- Model stability (PSI)



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

In [2]:
columns = [
    "checking_account", "duration", "credit_history", "purpose",
    "credit_amount", "savings_account", "employment_since",
    "installment_rate", "personal_status", "other_debtors",
    "residence_since", "property", "age", "other_installment_plans",
    "housing", "existing_credits", "job", "num_dependents",
    "telephone", "foreign_worker", "target"
]

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"

df = pd.read_csv(url, sep=' ', names=columns)
df.head()

,checking_account,duration,credit_history,purpose,credit_amount,savings_account,employment_since,installment_rate,personal_status,other_debtors,...,property,age,other_installment_plans,housing,existing_credits,job,num_dependents,telephone,foreign_worker,target
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


In [3]:
df['target'] = df['target'].apply(lambda x: 1 if x == 2 else 0)
df['target'].value_counts()

,count
target,
0,700
1,300


In [4]:
df = pd.get_dummies(df, drop_first=True)

In [5]:
X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [6]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = LogisticRegression(max_iter=5000, solver='liblinear')
model.fit(X_train, y_train)

LogisticRegression(max_iter=5000, solver='liblinear')

In [7]:
train_pred = model.predict_proba(X_train)[:,1]
test_pred = model.predict_proba(X_test)[:,1]

In [8]:
train_auc = roc_auc_score(y_train, train_pred)
test_auc = roc_auc_score(y_test, test_pred)

print("Train AUC:", train_auc)
print("Test AUC:", test_auc)

Train AUC: 0.8294955125269201
Test AUC: 0.817393133182607


## Overfitting Check

- If Train AUC is significantly greater than Test AUC it can be inferred as  Overfitting  
- If similar then
 Good generalization  

In [9]:
def calculate_psi(expected, actual, buckets=10):
    breakpoints = np.arange(0, buckets + 1) / buckets * 100

    expected_percents = np.percentile(expected, breakpoints)
    actual_percents = np.percentile(actual, breakpoints)

    psi_value = 0
    for i in range(len(expected_percents)-1):
        expected_count = ((expected >= expected_percents[i]) & (expected < expected_percents[i+1])).sum()
        actual_count = ((actual >= actual_percents[i]) & (actual < actual_percents[i+1])).sum()

        if expected_count == 0 or actual_count == 0:
            continue

        expected_ratio = expected_count / len(expected)
        actual_ratio = actual_count / len(actual)

        psi_value += (expected_ratio - actual_ratio) * np.log(expected_ratio / actual_ratio)

    return psi_value

psi = calculate_psi(train_pred, test_pred)
print("PSI:", psi)

PSI: 3.716726518777479e-05


## PSI Interpretation

- If PSI < 0.1 then Stable  
- 0.1–0.25 it implies Moderate shift  
- 0.25 means Unstable  

In [10]:
feature_shift = pd.DataFrame(X_train).mean() - pd.DataFrame(X_test).mean()
feature_shift.head()

,0
0,0.098069
1,0.152153
2,0.025133
3,-0.032395
4,-0.015753


## Model Validation Conclusion

- Compared train vs test performance  
- Checked overfitting  
- Evaluated model stability using PSI  

### Final Verdict
Model is stable and suitable for deployment if PSI is low and AUC is consistent.